In [170]:
import pandas as pd
from konlpy.tag import Mecab
from gensim import corpora
from gensim.models.ldamodel import LdaModel
import networkx as nx
import numpy as np
import tqdm

In [171]:
# --- 1단계: 데이터 준비 및 전처리 ---
print("1. 데이터 준비 및 전처리 시작...")


1. 데이터 준비 및 전처리 시작...


In [172]:
KINDS_PATH = '../data/interim/news/kinds_news.csv'
STOPWORD_PATH = '../data/raw/news/stopwords-ko.txt'

POSITIVE_PATH = '../data/raw/news/sentiment/positive.txt'
NEGATIVE_PATH = '../data/raw/news/sentiment/negative.txt'
NATURAL_PATH = '../data/raw/news/sentiment/natural.txt'
# 뉴스 데이터 읽기
df = pd.read_csv(KINDS_PATH)
#df = data.head(20) # 테스트용으로 상위 20개만 해봄

# 각종 TXT 파일 불러오기
def load_txt(PATH):
    with open(PATH, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]
# 불용어 불러오기
stopwords = load_txt(STOPWORD_PATH)
# 긍정, 부정, 중립 단어 불러오기
positive = load_txt(POSITIVE_PATH)
negative = load_txt(NEGATIVE_PATH)
natural = load_txt(NATURAL_PATH)

print(f"불용어 단어 예시 : {stopwords[:5]}...")
print(f"긍정 단어 예시 : {positive[:5]}...")
print(f"부정 단어 예시 : {negative[:5]}...")
print(f"중립 단어 예시 : {natural[:5]}...")

불용어 단어 예시 : ['가', '가까스로', '가령', '각', '각각']...
긍정 단어 예시 : ['활황', '급매물', '소진', '강세', '매수세']...
부정 단어 예시 : ['침체', '급매', '투매', '하락', '폭락']...
중립 단어 예시 : ['부동산', '아파트', '주택', '토지', '건물']...


In [173]:
# 명사 추출 + 불용어 제거 함수
mecab = Mecab()
def tokenize(text):
    return [word for word in mecab.nouns(text) 
            if len(word) > 1 and word not in stopwords]

# 토큰화 + 불용어 제거 적용
df['tokens'] = df['content'].apply(tokenize)

print(f"토큰 예시 : {df['tokens'][0]}")

토큰 예시 : ['신고', '신고', '신고', '과거', '패턴', '집값', '격차', '과거', '양극', '집값', '상승', '주목', '부동산', '거래', '분석']


In [174]:
# --- 2단계: 토픽 모델링 및 텍스트랭크 ---
print("\n2. 토픽 모델링 및 텍스트랭크 시작...")


2. 토픽 모델링 및 텍스트랭크 시작...


In [175]:
# 토픽 모델링 (LDA)
dictionary = corpora.Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(tokens) for tokens in df['tokens']]
lda_model = LdaModel(corpus, num_topics=8, id2word=dictionary, passes=15)

topics = lda_model.print_topics(num_words=30)
print(f"토픽 모델링 결과 예시 :\n {topics[0]}")

토픽 모델링 결과 예시 :
 (0, '0.067*"분양" + 0.054*"청약" + 0.046*"아파트" + 0.039*"가구" + 0.033*"공급" + 0.028*"부동산" + 0.023*"시장" + 0.022*"물량" + 0.017*"주택" + 0.016*"전국" + 0.015*"수도" + 0.015*"지역" + 0.012*"올해" + 0.012*"규제" + 0.011*"경쟁" + 0.011*"지방" + 0.010*"오피스텔" + 0.009*"순위" + 0.008*"업체" + 0.008*"일반" + 0.007*"지난해" + 0.007*"전망" + 0.006*"최근" + 0.006*"민간" + 0.006*"가운데" + 0.006*"평균" + 0.006*"수요자" + 0.006*"당첨" + 0.005*"시세" + 0.005*"예정"')


In [176]:
# 텍스트랭크
def text_rank_keywords(tokens):
    g = nx.Graph()
    for i in range(len(tokens) - 1):
        g.add_edge(tokens[i], tokens[i+1])
    pr = nx.pagerank(g, weight='weight')
    return sorted(pr, key=pr.get, reverse=True) # 상위 몇개를 포함할건가?는 논문에 없다

In [177]:
df['textrank_keywords'] = df['tokens'].apply(text_rank_keywords)
print("\n텍스트랭크 키워드:")
print(df[['content', 'textrank_keywords']].head())


텍스트랭크 키워드:
                                             content  \
0  신고가 건수 25배 차 벌어져 \n강남3구 3건중 1건 ‘신고가’ \n노도강 신고가...   
1  [KBS 광주]\n [앵커]\n\n 이사할 집을 계약하고 입주까지 마쳤는데 알고 보...   
2  올여름 분양시장에서 지역을 대표하는 ‘초고층 주거타운’ 내 신규 공급이 연이어 예고...   
3  올 1분기 전국 아파트 매매 거래량이 전년 동기에 비해 16.6% 증가한 것으로 나...   
4  재건축이 활발하게 진행되고 있는 서울 양천구 아파트값이 빠르게 치솟고 있다. 소규모...   

                                   textrank_keywords  
0      [집값, 과거, 거래, 부동산, 주목, 상승, 신고, 패턴, 격차, 양극, 분석]  
1                      [계약, 중고, 부동산, 사기, 거래, 임대, 이사]  
2  [지역, 주거, 초고층, 부촌, 시장, 수준, 조망권, 희소성, 집중, 강점, 상징...  
3  [거래량, 감소, 지역, 증가량, 부동산, 전국, 도시, 아파트, 지방, 매매, 영...  
4    [아파트, 최고, 기록, 지수, 매매, 소규모, 추진, 가치, 부동, 돌파, 재건축]  


In [178]:
# --- 3단계: 감성 사전 기반 감성 점수 산출 ---
print("\n3. 감성 사전 기반 감성 점수 산출 시작...")

def get_sentiment_score(tokens):
    pos_score = sum(1 for word in tokens if word in positive)
    neg_score = sum(1 for word in tokens if word in negative)
    nat_score = sum(1 for word in tokens if word in natural)
    total_words = len(tokens)
    if total_words == 0:
        return 0
    return (pos_score - neg_score) / total_words

df['sentiment_score'] = df['tokens'].apply(get_sentiment_score)

print("\n감성 사전 기반 감성 점수:")
print(df[['content', 'sentiment_score']].head())


3. 감성 사전 기반 감성 점수 산출 시작...

감성 사전 기반 감성 점수:
                                             content  sentiment_score
0  신고가 건수 25배 차 벌어져 \n강남3구 3건중 1건 ‘신고가’ \n노도강 신고가...         0.066667
1  [KBS 광주]\n [앵커]\n\n 이사할 집을 계약하고 입주까지 마쳤는데 알고 보...        -0.083333
2  올여름 분양시장에서 지역을 대표하는 ‘초고층 주거타운’ 내 신규 공급이 연이어 예고...         0.086957
3  올 1분기 전국 아파트 매매 거래량이 전년 동기에 비해 16.6% 증가한 것으로 나...         0.117647
4  재건축이 활발하게 진행되고 있는 서울 양천구 아파트값이 빠르게 치솟고 있다. 소규모...         0.307692


In [179]:
# --- 4단계: 월별 감성 지수 산출 및 예측 모델 통합 ---
print("\n4. 월별 감성 지수 산출 및 예측 모델 통합...")


4. 월별 감성 지수 산출 및 예측 모델 통합...


In [181]:
# datetime 변환
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
df['month'] = df['date'].dt.to_period('M')

In [231]:
monthly_sentiment = df.groupby('month')['sentiment_score'].mean().reset_index()
monthly_sentiment['month'] = monthly_sentiment['month'].astype(str)

# monthly_sentiment month 컬럼도 period[M]로 변환
monthly_sentiment['month'] = pd.to_datetime(monthly_sentiment['month']).dt.to_period('M')


In [232]:
print("\n월별 감성 지수:")
print(monthly_sentiment)


월별 감성 지수:
      month  sentiment_score
0   2020-07        -0.020122
1   2020-08        -0.020081
2   2020-09        -0.011923
3   2020-10        -0.006215
4   2020-11        -0.017122
5   2020-12        -0.007766
6   2021-01        -0.004556
7   2021-02         0.013726
8   2021-03         0.004142
9   2021-04         0.001894
10  2021-05        -0.004065
11  2021-06         0.003813
12  2021-07         0.002564
13  2021-08        -0.016914
14  2021-09         0.006719
15  2021-10        -0.015591
16  2021-11        -0.016538
17  2021-12        -0.019967
18  2022-01        -0.022945
19  2022-02        -0.018046
20  2022-03        -0.016491
21  2022-04        -0.007278
22  2022-05        -0.002780
23  2022-06        -0.019746
24  2022-07        -0.039413
25  2022-08        -0.024150
26  2022-09        -0.046294
27  2022-10        -0.056421
28  2022-11        -0.057009
29  2022-12        -0.056531
30  2023-01        -0.052522
31  2023-02        -0.030479
32  2023-03        -0.027509
33 

In [233]:
SALE_PATH = '../data/interim/apt/apt_with_long_lat.csv'
sale = pd.read_csv(SALE_PATH)

In [234]:
# datetime 변환
sale['계약일자'] = pd.to_datetime(sale['계약일자'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
sale['month'] = sale['계약일자'].dt.to_period('M')
drop_col = ['단지명','도로명','계약일자','경도','위도']

In [235]:
sale.columns

Index(['단지명', '전용면적(㎡)', '층', '건축년도', '도로명', '면적당 단가(만원)', '아파트 나이', '계약일자',
       'alpha', '경도', '위도', 'month'],
      dtype='object')

In [236]:
sale.drop(drop_col, axis=1, inplace=True)

In [237]:
sale.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,month
0,59.91,10,1998,6.766156,22,0.266667,2020-07
1,59.77,7,1996,7.499383,24,1.000000,2020-07
2,84.83,6,2013,7.401580,7,1.000000,2020-07
3,59.75,13,2016,7.066081,4,1.000000,2020-07
4,49.94,7,1989,6.967225,31,0.000000,2020-07


In [238]:
merged_df = pd.merge(sale, monthly_sentiment, on='month', how='left')

In [242]:
merged_df.drop('month', axis=1, inplace=True)

In [245]:
merged_df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,sentiment_score
0,59.91,10,1998,6.766156,22,0.266667,-0.020122
1,59.77,7,1996,7.499383,24,1.000000,-0.020122
2,84.83,6,2013,7.401580,7,1.000000,-0.020122
3,59.75,13,2016,7.066081,4,1.000000,-0.020122
4,49.94,7,1989,6.967225,31,0.000000,-0.020122


In [259]:
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
import numpy as np


In [260]:

# 특성과 타깃 분리
X = merged_df[['전용면적(㎡)','층','건축년도','아파트 나이','alpha','sentiment_score']]
y = merged_df['면적당 단가(만원)']

# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(124,64,32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')

# MAE는 음수로 반환되므로 양수로 변환
mae_scores = -scores

print("10-fold CV MAE scores:", mae_scores)
print("Mean MAE:", np.mean(mae_scores))

10-fold CV MAE scores: [0.33347935 0.33173509 0.32629946 0.33297623 0.32915204 0.35780532
 0.32653338 0.34020012 0.33243182 0.37574076]
Mean MAE: 0.33863535689527313


In [261]:
sale

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha
0,59.91,10,1998,6.766156,22,0.266667
1,59.77,7,1996,7.499383,24,1.000000
2,84.83,6,2013,7.401580,7,1.000000
3,59.75,13,2016,7.066081,4,1.000000
4,49.94,7,1989,6.967225,31,0.000000
...,...,...,...,...,...,...
23568,54.34,2,1995,5.908227,30,0.000000
23569,97.21,8,2006,7.406056,19,0.366667
23570,23.70,11,2019,7.034407,6,0.800000
23571,22.20,10,2002,7.139868,23,0.233333


In [262]:

# 특성과 타깃 분리
X = sale[['전용면적(㎡)','층','건축년도','아파트 나이','alpha']]
y = sale['면적당 단가(만원)']

# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(124,64,32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')

# MAE는 음수로 반환되므로 양수로 변환
mae_scores = -scores

print("10-fold CV MAE scores:", mae_scores)
print("Mean MAE:", np.mean(mae_scores))

10-fold CV MAE scores: [0.32884437 0.33208562 0.3336433  0.33051902 0.32690495 0.33568008
 0.33561399 0.33497199 0.32983879 0.35079056]
Mean MAE: 0.33388926646135947
